# Student Mental Health Dataset - Exploratory Data Analysis (EDA)

This notebook performs comprehensive exploratory data analysis on the student mental health dataset to understand data characteristics, missing values, correlations, and data quality before conducting analysis.

## Analysis Objectives:
- Understand dataset structure and basic characteristics
- Identify and analyze missing values patterns
- Explore data distributions and relationships
- Assess data quality and identify potential issues
- Generate insights for informed analysis decisions

## 1. Import Required Libraries

In [ ]:
# Essential libraries for data analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys
from pathlib import Path

# Missingno for missing values visualization
import missingno as msno

# Statistical libraries
from scipy import stats
from scipy.stats import pearsonr, chi2_contingency

# Configure plotting
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
warnings.filterwarnings('ignore')

# Add project root to path for imports
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

print("Libraries imported successfully!")
print(f"Working directory: {project_root}")
print(f"Python version: {sys.version}")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Matplotlib version: {plt.matplotlib.__version__}")
print(f"Seaborn version: {sns.__version__}")

## 2. Load and Inspect the Dataset

In [ ]:
# Load data using the project's repository pattern
from db.repository import StudentMentalHealthRepository

# Initialize data repository and load data
try:
    data_repo = StudentMentalHealthRepository("data/student_mental_health.db")
    df = data_repo.get_all_data()
    
    print(f"✅ Dataset loaded successfully!")
    print(f"Dataset shape: {df.shape}")
    print(f"Total students: {len(df)}")
    
except Exception as e:
    print(f"❌ Error loading data: {str(e)}")
    df = None

In [ ]:
# Initial data inspection
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)

if df is not None:
    # Basic information
    print(f"\n📊 Dataset Dimensions:")
    print(f"   • Rows: {df.shape[0]:,}")
    print(f"   • Columns: {df.shape[1]}")
    
    # Display first few rows
    print(f"\n🔍 First 5 rows:")
    print("-" * 50)
    display(df.head())
    
    # Display last few rows
    print(f"\n🔚 Last 5 rows:")
    print("-" * 50)
    display(df.tail())
    
    # Column names
    print(f"\n📝 Column Names ({len(df.columns)} total):")
    print("-" * 50)
    for i, col in enumerate(df.columns, 1):
        print(f"{i:2d}. {col}")
else:
    print("❌ Cannot proceed with analysis - dataset not loaded")

## 3. Basic Dataset Information

In [ ]:
# Comprehensive dataset information
if df is not None:
    print("=" * 60)
    print("DETAILED DATASET INFORMATION")
    print("=" * 60)
    
    # DataFrame info
    print("\n📋 DataFrame Info:")
    print("-" * 50)
    df.info(memory_usage='deep')
    
    # Memory usage analysis
    print(f"\n💾 Memory Usage Analysis:")
    print("-" * 50)
    memory_usage = df.memory_usage(deep=True)
    total_memory = memory_usage.sum()
    print(f"   • Total memory usage: {total_memory / 1024**2:.2f} MB")
    print(f"   • Average memory per row: {total_memory / len(df):.0f} bytes")
    
    # Top memory consuming columns
    print(f"\n📊 Top 5 Memory Consuming Columns:")
    print("-" * 50)
    top_memory = memory_usage.nlargest(6)[1:]  # Exclude Index
    for col, usage in top_memory.items():
        print(f"   • {col:<20}: {usage / 1024:.2f} KB ({usage/total_memory*100:.1f}%)")
        
else:
    print("❌ Cannot analyze dataset information - data not available")

## 4. Missing Values Analysis with Missingno

In [ ]:
# Missing values analysis - numerical summary
if df is not None:
    print("=" * 60)
    print("MISSING VALUES ANALYSIS")
    print("=" * 60)
    
    # Calculate missing values
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100
    
    # Create missing values summary
    missing_summary = pd.DataFrame({
        'Column': missing_data.index,
        'Missing_Count': missing_data.values,
        'Missing_Percentage': missing_percent.values
    }).sort_values('Missing_Count', ascending=False)
    
    # Display summary
    print(f"\n📊 Missing Values Summary:")
    print("-" * 50)
    print(f"Total missing values in dataset: {missing_data.sum():,}")
    print(f"Percentage of total dataset missing: {(missing_data.sum()/(len(df)*len(df.columns)))*100:.2f}%")
    
    # Show columns with missing values
    columns_with_missing = missing_summary[missing_summary['Missing_Count'] > 0]
    
    if len(columns_with_missing) > 0:
        print(f"\n🚨 Columns with missing values ({len(columns_with_missing)} total):")
        print("-" * 50)
        display(columns_with_missing)
    else:
        print(f"\n✅ No missing values found in the dataset!")
        
    # Show columns without missing values
    columns_complete = missing_summary[missing_summary['Missing_Count'] == 0]
    print(f"\n✅ Complete columns ({len(columns_complete)} total):")
    print("-" * 50)
    if len(columns_complete) <= 10:
        for col in columns_complete['Column']:
            print(f"   • {col}")
    else:
        for col in columns_complete['Column'][:10]:
            print(f"   • {col}")
        print(f"   ... and {len(columns_complete) - 10} more columns")
        
else:
    print("❌ Cannot analyze missing values - data not available")

In [ ]:
# Missing values visualization using missingno
if df is not None:
    print("🎨 Missing Values Visualizations:")
    print("-" * 50)
    
    # Set up the plotting space
    fig, axes = plt.subplots(2, 2, figsize=(20, 16))
    
    # 1. Missing values bar chart
    ax1 = plt.subplot(2, 2, 1)
    msno.bar(df, ax=ax1)
    plt.title('Missing Values Bar Chart\n(Shows count of non-null values per column)', 
              fontsize=12, fontweight='bold', pad=20)
    
    # 2. Missing values matrix
    ax2 = plt.subplot(2, 2, 2)
    msno.matrix(df, ax=ax2)
    plt.title('Missing Values Matrix\n(White lines indicate missing values)', 
              fontsize=12, fontweight='bold', pad=20)
    
    # 3. Missing values heatmap (correlations)
    ax3 = plt.subplot(2, 2, 3)
    msno.heatmap(df, ax=ax3)
    plt.title('Missing Values Heatmap\n(Correlation between missing values)', 
              fontsize=12, fontweight='bold', pad=20)
    
    # 4. Missing values dendrogram (if applicable)
    ax4 = plt.subplot(2, 2, 4)
    try:
        msno.dendrogram(df, ax=ax4)
        plt.title('Missing Values Dendrogram\n(Hierarchical clustering of missing values)', 
                  fontsize=12, fontweight='bold', pad=20)
    except Exception as e:
        ax4.text(0.5, 0.5, f'Dendrogram not available\n({str(e)})', 
                ha='center', va='center', transform=ax4.transAxes)
        ax4.set_title('Missing Values Dendrogram - Not Available')
    
    plt.tight_layout()
    plt.show()
    
    print("\n📋 Interpretation Guide:")
    print("   • Bar Chart: Shows the count of non-null values for each column")
    print("   • Matrix: Each row is a data point, white lines show missing values")
    print("   • Heatmap: Shows correlation between missingness of different variables")
    print("   • Dendrogram: Shows how missing values cluster together")
    
else:
    print("❌ Cannot create missing values visualizations - data not available")

## 5. Data Types and Memory Usage

In [ ]:
# Data types analysis
if df is not None:
    print("=" * 60)
    print("DATA TYPES AND MEMORY ANALYSIS")
    print("=" * 60)
    
    # Create comprehensive data types summary
    dtypes_info = []
    for col in df.columns:
        col_info = {
            'Column': col,
            'Data_Type': str(df[col].dtype),
            'Non_Null_Count': df[col].count(),
            'Null_Count': df[col].isnull().sum(),
            'Unique_Values': df[col].nunique(),
            'Memory_Usage_KB': df[col].memory_usage(deep=True) / 1024
        }
        dtypes_info.append(col_info)
    
    dtypes_df = pd.DataFrame(dtypes_info)
    
    print(f"\n📊 Data Types Summary:")
    print("-" * 50)
    dtype_counts = dtypes_df['Data_Type'].value_counts()
    for dtype, count in dtype_counts.items():
        print(f"   • {dtype:<12}: {count:2d} columns ({count/len(df.columns)*100:.1f}%)")
    
    print(f"\n📋 Detailed Column Information:")
    print("-" * 50)
    display(dtypes_df.sort_values('Memory_Usage_KB', ascending=False))
    
    # Identify potential optimization opportunities
    print(f"\n💡 Optimization Opportunities:")
    print("-" * 50)
    
    # Object columns that might be categorical
    object_cols = dtypes_df[dtypes_df['Data_Type'] == 'object']
    categorical_candidates = object_cols[object_cols['Unique_Values'] < 20]
    
    if len(categorical_candidates) > 0:
        print(f"   🎯 Potential categorical conversions ({len(categorical_candidates)} columns):")
        for _, row in categorical_candidates.iterrows():
            memory_saving = row['Memory_Usage_KB'] * 0.7  # Rough estimate
            print(f"     • {row['Column']:<20}: {row['Unique_Values']:2d} unique values (~{memory_saving:.1f} KB savings)")
    
    # High cardinality object columns
    high_cardinality = object_cols[object_cols['Unique_Values'] > 100]
    if len(high_cardinality) > 0:
        print(f"   ⚠️  High cardinality object columns ({len(high_cardinality)} columns):")
        for _, row in high_cardinality.iterrows():
            print(f"     • {row['Column']:<20}: {row['Unique_Values']:4d} unique values")
            
else:
    print("❌ Cannot analyze data types - data not available")

## 6. Descriptive Statistics

In [ ]:
# Comprehensive descriptive statistics
if df is not None:
    print("=" * 60)
    print("DESCRIPTIVE STATISTICS")
    print("=" * 60)
    
    # Separate numerical and categorical columns
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    print(f"\n📊 Column Classification:")
    print("-" * 50)
    print(f"   • Numerical columns: {len(numerical_cols)}")
    print(f"   • Categorical columns: {len(categorical_cols)}")
    
    # Numerical statistics
    if numerical_cols:
        print(f"\n🔢 Numerical Variables Statistics:")
        print("-" * 50)
        numerical_stats = df[numerical_cols].describe()
        
        # Add additional statistics
        numerical_stats.loc['variance'] = df[numerical_cols].var()
        numerical_stats.loc['skewness'] = df[numerical_cols].skew()
        numerical_stats.loc['kurtosis'] = df[numerical_cols].kurtosis()
        
        display(numerical_stats.round(3))
        
        # Identify potential outliers using IQR method
        print(f"\n⚠️  Potential Outliers (IQR Method):")
        print("-" * 50)
        outlier_summary = []
        for col in numerical_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
            outlier_summary.append({
                'Column': col,
                'Outliers_Count': len(outliers),
                'Outliers_Percentage': len(outliers) / len(df) * 100,
                'Lower_Bound': lower_bound,
                'Upper_Bound': upper_bound
            })
        
        outlier_df = pd.DataFrame(outlier_summary)
        outlier_df = outlier_df[outlier_df['Outliers_Count'] > 0].sort_values('Outliers_Percentage', ascending=False)
        if len(outlier_df) > 0:
            display(outlier_df.round(3))
        else:
            print("   ✅ No outliers detected using IQR method")
    else:
        print(f"\n🔢 No numerical columns found")
        
else:
    print("❌ Cannot generate descriptive statistics - data not available")

In [ ]:
# Categorical variables analysis
if df is not None and categorical_cols:
    print(f"\n🏷️  Categorical Variables Analysis:")
    print("-" * 50)
    
    categorical_summary = []
    for col in categorical_cols:
        col_summary = {
            'Column': col,
            'Unique_Count': df[col].nunique(),
            'Most_Common': df[col].mode().iloc[0] if not df[col].mode().empty else 'N/A',
            'Most_Common_Count': df[col].value_counts().iloc[0] if len(df[col].value_counts()) > 0 else 0,
            'Most_Common_Percentage': (df[col].value_counts().iloc[0] / len(df) * 100) if len(df[col].value_counts()) > 0 else 0
        }
        categorical_summary.append(col_summary)
    
    categorical_df = pd.DataFrame(categorical_summary)
    display(categorical_df)
    
    # Detailed analysis for key categorical variables (show top categories)
    print(f"\n📊 Top Categories for Each Variable:")
    print("-" * 50)
    for col in categorical_cols[:5]:  # Show first 5 categorical columns
        print(f"\n• {col}:")
        value_counts = df[col].value_counts()
        for i, (value, count) in enumerate(value_counts.head().items()):
            percentage = count / len(df) * 100
            print(f"  {i+1}. {value}: {count:,} ({percentage:.1f}%)")
        if len(value_counts) > 5:
            print(f"  ... and {len(value_counts) - 5} more categories")
            
elif df is not None:
    print(f"\n🏷️  No categorical columns found")

## 7. Correlation Analysis

In [ ]:
# Correlation analysis for numerical variables
if df is not None and numerical_cols and len(numerical_cols) > 1:
    print("=" * 60)
    print("CORRELATION ANALYSIS")
    print("=" * 60)
    
    # Calculate correlation matrix
    correlation_matrix = df[numerical_cols].corr()
    
    print(f"\n📊 Correlation Matrix ({len(numerical_cols)} numerical variables):")
    print("-" * 50)
    
    # Create correlation heatmap
    plt.figure(figsize=(12, 10))
    mask = np.triu(correlation_matrix.corr())
    sns.heatmap(correlation_matrix, 
                annot=True, 
                cmap='RdBu_r', 
                center=0,
                square=True,
                mask=mask,
                cbar_kws={'label': 'Correlation Coefficient'},
                fmt='.3f')
    plt.title('Correlation Matrix Heatmap\n(Upper triangle masked)', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    
    # Identify strong correlations
    print(f"\n🔍 Strong Correlations (|r| > 0.5):")
    print("-" * 50)
    
    strong_correlations = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_value = correlation_matrix.iloc[i, j]
            if abs(corr_value) > 0.5:
                strong_correlations.append({
                    'Variable_1': correlation_matrix.columns[i],
                    'Variable_2': correlation_matrix.columns[j],
                    'Correlation': corr_value,
                    'Strength': 'Very Strong' if abs(corr_value) > 0.8 else 'Strong',
                    'Direction': 'Positive' if corr_value > 0 else 'Negative'
                })
    
    if strong_correlations:
        strong_corr_df = pd.DataFrame(strong_correlations).sort_values('Correlation', key=abs, ascending=False)
        display(strong_corr_df)
    else:
        print("   ℹ️  No strong correlations (|r| > 0.5) found between numerical variables")
    
    # Summary statistics for correlations
    corr_values = correlation_matrix.values
    corr_values = corr_values[np.triu_indices_from(corr_values, k=1)]  # Upper triangle, excluding diagonal
    
    print(f"\n📈 Correlation Statistics:")
    print("-" * 50)
    print(f"   • Mean absolute correlation: {np.mean(np.abs(corr_values)):.3f}")
    print(f"   • Maximum correlation: {np.max(corr_values):.3f}")
    print(f"   • Minimum correlation: {np.min(corr_values):.3f}")
    print(f"   • Standard deviation: {np.std(corr_values):.3f}")
    
elif df is not None:
    print("=" * 60)
    print("CORRELATION ANALYSIS")
    print("=" * 60)
    print(f"\n⚠️  Insufficient numerical variables for correlation analysis")
    print(f"    (Found {len(numerical_cols) if 'numerical_cols' in locals() else 0} numerical columns, need at least 2)")
else:
    print("❌ Cannot perform correlation analysis - data not available")

## 8. Categorical Variables Analysis

In [ ]:
# Detailed categorical variables analysis and visualization
if df is not None and categorical_cols:
    print("=" * 60)
    print("CATEGORICAL VARIABLES DEEP DIVE")
    print("=" * 60)
    
    # Select key categorical variables for visualization (first 6 or all if fewer)
    key_categorical = categorical_cols[:6]
    
    # Create subplots for categorical variables
    n_cols = 3
    n_rows = (len(key_categorical) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for i, col in enumerate(key_categorical):
        row = i // n_cols
        col_idx = i % n_cols
        ax = axes[row, col_idx]
        
        # Get value counts
        value_counts = df[col].value_counts()
        
        # Create bar plot
        if len(value_counts) <= 10:  # If not too many categories
            bars = ax.bar(range(len(value_counts)), value_counts.values, color=plt.cm.Set3(np.linspace(0, 1, len(value_counts))))
            ax.set_xticks(range(len(value_counts)))
            ax.set_xticklabels(value_counts.index, rotation=45, ha='right')
            
            # Add value labels on bars
            for j, (bar, count) in enumerate(zip(bars, value_counts.values)):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                       str(count), ha='center', va='bottom', fontsize=10)
        else:
            # Too many categories, show top 10
            top_10 = value_counts.head(10)
            bars = ax.bar(range(len(top_10)), top_10.values, color=plt.cm.Set3(np.linspace(0, 1, len(top_10))))
            ax.set_xticks(range(len(top_10)))
            ax.set_xticklabels(top_10.index, rotation=45, ha='right')
            ax.set_title(f'{col} (Top 10 Categories)')
            
        ax.set_title(f'{col}\n({len(value_counts)} unique values)', fontsize=12, fontweight='bold')
        ax.set_ylabel('Count')
        ax.grid(True, alpha=0.3, axis='y')
    
    # Hide empty subplots
    for i in range(len(key_categorical), n_rows * n_cols):
        row = i // n_cols
        col_idx = i % n_cols
        axes[row, col_idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Data quality checks for categorical variables
    print(f"\n🔍 Categorical Variables Quality Check:")
    print("-" * 50)
    
    quality_issues = []
    for col in categorical_cols:
        issues = []
        
        # Check for potential inconsistencies (case sensitivity, whitespace)
        values = df[col].astype(str).str.lower().str.strip()
        original_unique = df[col].nunique()
        cleaned_unique = values.nunique()
        
        if cleaned_unique < original_unique:
            issues.append(f"Case/whitespace inconsistencies ({original_unique} -> {cleaned_unique})")
        
        # Check for very low frequency categories
        value_counts = df[col].value_counts()
        rare_categories = value_counts[value_counts == 1]
        if len(rare_categories) > 0:
            issues.append(f"{len(rare_categories)} singleton categories")
            
        # Check for potential missing value representations
        missing_representations = ['missing', 'unknown', 'n/a', 'na', 'none', '', ' ']
        actual_values = df[col].astype(str).str.lower().str.strip()
        found_missing_repr = [repr_val for repr_val in missing_representations if repr_val in actual_values.values]
        if found_missing_repr:
            issues.append(f"Potential missing value representations: {found_missing_repr}")
        
        if issues:
            quality_issues.append({'Column': col, 'Issues': '; '.join(issues)})
    
    if quality_issues:
        print("   ⚠️  Quality issues found:")
        for issue in quality_issues:
            print(f"     • {issue['Column']}: {issue['Issues']}")
    else:
        print("   ✅ No quality issues detected in categorical variables")
        
elif df is not None:
    print("=" * 60)
    print("CATEGORICAL VARIABLES ANALYSIS")
    print("=" * 60)
    print(f"\n📊 No categorical variables found in the dataset")
else:
    print("❌ Cannot analyze categorical variables - data not available")

## 9. Numerical Variables Distribution

In [ ]:
# Distribution analysis for numerical variables
if df is not None and numerical_cols:
    print("=" * 60)
    print("NUMERICAL VARIABLES DISTRIBUTION ANALYSIS")
    print("=" * 60)
    
    # Create distribution plots
    n_cols = 2
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 6*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    elif len(numerical_cols) == 1:
        axes = np.array([axes])
    
    for i, col in enumerate(numerical_cols):
        row = i // n_cols
        col_idx = i % n_cols
        ax = axes[row, col_idx]
        
        # Create histogram with density curve
        df[col].hist(bins=30, alpha=0.7, color='skyblue', edgecolor='black', ax=ax)
        
        # Add density curve
        try:
            df[col].plot.density(ax=ax, color='red', linewidth=2)
        except:
            pass  # Skip if density plot fails
        
        ax.set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Frequency / Density')
        ax.grid(True, alpha=0.3)
        
        # Add statistics as text
        stats_text = f'Mean: {df[col].mean():.2f}\n'
        stats_text += f'Median: {df[col].median():.2f}\n'
        stats_text += f'Std: {df[col].std():.2f}\n'
        stats_text += f'Skew: {df[col].skew():.2f}'
        
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8), fontsize=9)
    
    # Hide empty subplots
    for i in range(len(numerical_cols), n_rows * n_cols):
        row = i // n_cols
        col_idx = i % n_cols
        axes[row, col_idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Distribution analysis summary
    print(f"\n📊 Distribution Analysis Summary:")
    print("-" * 50)
    
    distribution_summary = []
    for col in numerical_cols:
        skewness = df[col].skew()
        kurtosis = df[col].kurtosis()
        
        # Classify distribution shape
        if abs(skewness) < 0.5:
            shape = "Approximately Normal"
        elif skewness > 0.5:
            shape = "Right Skewed"
        elif skewness < -0.5:
            shape = "Left Skewed"
        else:
            shape = "Moderate Skew"
            
        # Classify tail heaviness
        if kurtosis > 3:
            tail = "Heavy Tails"
        elif kurtosis < -1:
            tail = "Light Tails"
        else:
            tail = "Normal Tails"
        
        distribution_summary.append({
            'Variable': col,
            'Mean': df[col].mean(),
            'Median': df[col].median(),
            'Std_Dev': df[col].std(),
            'Skewness': skewness,
            'Kurtosis': kurtosis,
            'Shape': shape,
            'Tails': tail
        })
    
    dist_df = pd.DataFrame(distribution_summary)
    display(dist_df.round(3))
    
elif df is not None:
    print("=" * 60)
    print("NUMERICAL VARIABLES DISTRIBUTION ANALYSIS")
    print("=" * 60)
    print(f"\n📊 No numerical variables found for distribution analysis")
else:
    print("❌ Cannot analyze distributions - data not available")

## 10. Outlier Detection

In [ ]:
# Comprehensive outlier detection
if df is not None and numerical_cols:
    print("=" * 60)
    print("OUTLIER DETECTION ANALYSIS")
    print("=" * 60)
    
    # Create box plots for outlier visualization
    n_cols = 2
    n_rows = (len(numerical_cols) + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5*n_rows))
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    elif len(numerical_cols) == 1:
        axes = np.array([axes])
    
    for i, col in enumerate(numerical_cols):
        row = i // n_cols
        col_idx = i % n_cols
        ax = axes[row, col_idx]
        
        # Create box plot
        box_plot = ax.boxplot(df[col].dropna(), patch_artist=True, 
                             boxprops=dict(facecolor='lightblue', alpha=0.7),
                             medianprops=dict(color='red', linewidth=2))
        
        ax.set_title(f'Box Plot - {col}', fontsize=12, fontweight='bold')
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3, axis='y')
    
    # Hide empty subplots
    for i in range(len(numerical_cols), n_rows * n_cols):
        row = i // n_cols
        col_idx = i % n_cols
        axes[row, col_idx].set_visible(False)
    
    plt.tight_layout()
    plt.show()
    
    # Detailed outlier analysis using multiple methods
    print(f"\n🔍 Outlier Detection Results:")
    print("-" * 50)
    
    outlier_results = []
    
    for col in numerical_cols:
        col_data = df[col].dropna()
        
        # Method 1: IQR Method
        Q1 = col_data.quantile(0.25)
        Q3 = col_data.quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        iqr_outliers = col_data[(col_data < lower_bound) | (col_data > upper_bound)]
        
        # Method 2: Z-Score Method (|z| > 3)
        z_scores = np.abs(stats.zscore(col_data))
        z_outliers = col_data[z_scores > 3]
        
        # Method 3: Modified Z-Score Method (using median)
        median = np.median(col_data)
        mad = np.median(np.abs(col_data - median))
        modified_z_scores = 0.6745 * (col_data - median) / mad
        modified_z_outliers = col_data[np.abs(modified_z_scores) > 3.5]
        
        outlier_results.append({
            'Variable': col,
            'IQR_Outliers': len(iqr_outliers),
            'IQR_Percentage': len(iqr_outliers) / len(col_data) * 100,
            'Z_Score_Outliers': len(z_outliers),
            'Z_Score_Percentage': len(z_outliers) / len(col_data) * 100,
            'Modified_Z_Outliers': len(modified_z_outliers),
            'Modified_Z_Percentage': len(modified_z_outliers) / len(col_data) * 100,
            'Range': f"{col_data.min():.2f} to {col_data.max():.2f}"
        })
    
    outlier_df = pd.DataFrame(outlier_results)
    display(outlier_df.round(3))
    
    # Summary recommendations
    print(f"\n💡 Outlier Analysis Recommendations:")
    print("-" * 50)
    
    for _, row in outlier_df.iterrows():
        recommendations = []
        
        if row['IQR_Percentage'] > 5:
            recommendations.append("High outlier rate (IQR method) - investigate data quality")
        elif row['IQR_Percentage'] > 1:
            recommendations.append("Moderate outliers detected - consider transformation")
        
        if row['Z_Score_Outliers'] > 0:
            recommendations.append("Extreme values detected (Z-score) - review for errors")
            
        if not recommendations:
            recommendations.append("Low outlier rate - data appears clean")
        
        print(f"   • {row['Variable']}: {'; '.join(recommendations)}")
        
elif df is not None:
    print("=" * 60)
    print("OUTLIER DETECTION ANALYSIS")
    print("=" * 60)
    print(f"\n📊 No numerical variables available for outlier detection")
else:
    print("❌ Cannot perform outlier detection - data not available")

## 11. Data Quality Assessment

In [ ]:
# Comprehensive data quality assessment
if df is not None:
    print("=" * 60)
    print("DATA QUALITY ASSESSMENT")
    print("=" * 60)
    
    quality_report = {
        'Dataset Size': len(df),
        'Number of Columns': len(df.columns),
        'Memory Usage (MB)': df.memory_usage(deep=True).sum() / 1024**2,
        'Missing Values': df.isnull().sum().sum(),
        'Missing Percentage': (df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100,
        'Duplicate Rows': df.duplicated().sum(),
        'Duplicate Percentage': (df.duplicated().sum() / len(df)) * 100
    }
    
    print(f"\n📊 Overall Data Quality Metrics:")
    print("-" * 50)
    for metric, value in quality_report.items():
        if 'Percentage' in metric:
            print(f"   • {metric:<20}: {value:.2f}%")
        elif 'MB' in metric:
            print(f"   • {metric:<20}: {value:.2f} MB")
        else:
            print(f"   • {metric:<20}: {value:,}")
    
    # Check for duplicate rows
    if quality_report['Duplicate Rows'] > 0:
        print(f"\n🔍 Duplicate Rows Analysis:")
        print("-" * 50)
        duplicates = df[df.duplicated(keep=False)]
        print(f"   • Total duplicate rows: {len(duplicates):,}")
        print(f"   • First few duplicates:")
        display(duplicates.head())
    else:
        print(f"\n✅ No duplicate rows found")
    
    # Data consistency checks
    print(f"\n🔍 Data Consistency Checks:")
    print("-" * 50)
    
    consistency_issues = []
    
    # Check for mixed data types in object columns
    for col in df.select_dtypes(include=['object']).columns:
        try:
            # Try to detect numeric values in object columns
            numeric_mask = pd.to_numeric(df[col], errors='coerce').notna()
            if numeric_mask.sum() > 0 and numeric_mask.sum() < len(df[col]):
                consistency_issues.append(f"{col}: Mixed data types detected")
        except:
            pass
    
    # Check for unusual string patterns
    for col in df.select_dtypes(include=['object']).columns:
        if df[col].dtype == 'object':
            # Check for leading/trailing whitespace
            if df[col].astype(str).str.strip().ne(df[col].astype(str)).any():
                consistency_issues.append(f"{col}: Leading/trailing whitespace found")
            
            # Check for case inconsistencies
            unique_values = df[col].dropna().astype(str)
            lower_unique = unique_values.str.lower().unique()
            if len(unique_values.unique()) > len(lower_unique):
                consistency_issues.append(f"{col}: Case inconsistencies detected")
    
    if consistency_issues:
        print("   ⚠️  Consistency issues found:")
        for issue in consistency_issues:
            print(f"     • {issue}")
    else:
        print("   ✅ No major consistency issues detected")
    
    # Data completeness by column
    print(f"\n📊 Data Completeness by Column:")
    print("-" * 50)
    completeness = []
    for col in df.columns:
        non_null_count = df[col].count()
        completeness_pct = (non_null_count / len(df)) * 100
        completeness.append({
            'Column': col,
            'Non_Null_Count': non_null_count,
            'Completeness_Percentage': completeness_pct,
            'Quality_Level': 'Excellent' if completeness_pct >= 95 else 
                            'Good' if completeness_pct >= 90 else 
                            'Fair' if completeness_pct >= 75 else 'Poor'
        })
    
    completeness_df = pd.DataFrame(completeness).sort_values('Completeness_Percentage', ascending=False)
    display(completeness_df.round(2))
    
    # Final data quality summary
    print(f"\n📋 Data Quality Summary:")
    print("-" * 50)
    
    excellent_cols = len(completeness_df[completeness_df['Quality_Level'] == 'Excellent'])
    good_cols = len(completeness_df[completeness_df['Quality_Level'] == 'Good'])
    fair_cols = len(completeness_df[completeness_df['Quality_Level'] == 'Fair'])
    poor_cols = len(completeness_df[completeness_df['Quality_Level'] == 'Poor'])
    
    print(f"   • Excellent quality columns (≥95% complete): {excellent_cols}")
    print(f"   • Good quality columns (90-94% complete): {good_cols}")
    print(f"   • Fair quality columns (75-89% complete): {fair_cols}")
    print(f"   • Poor quality columns (<75% complete): {poor_cols}")
    
    # Overall quality score
    avg_completeness = completeness_df['Completeness_Percentage'].mean()
    if avg_completeness >= 95:
        quality_grade = "A (Excellent)"
    elif avg_completeness >= 90:
        quality_grade = "B (Good)"
    elif avg_completeness >= 75:
        quality_grade = "C (Fair)"
    else:
        quality_grade = "D (Poor)"
    
    print(f"\n🏆 Overall Data Quality Grade: {quality_grade}")
    print(f"    Average completeness: {avg_completeness:.1f}%")
    
else:
    print("❌ Cannot perform data quality assessment - data not available")

## 📋 EDA Summary and Recommendations

This comprehensive exploratory data analysis provides insights into the student mental health dataset structure, quality, and patterns. Key findings and recommendations for further analysis are summarized below.

In [ ]:
# Final EDA Summary and Recommendations
if df is not None:
    print("=" * 80)
    print("🎯 EXPLORATORY DATA ANALYSIS - FINAL SUMMARY & RECOMMENDATIONS")
    print("=" * 80)
    
    print(f"\n📊 DATASET OVERVIEW:")
    print(f"   • Dataset Size: {len(df):,} rows × {len(df.columns)} columns")
    print(f"   • Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Numerical Variables: {len(df.select_dtypes(include=[np.number]).columns)}")
    print(f"   • Categorical Variables: {len(df.select_dtypes(include=['object', 'category']).columns)}")
    
    print(f"\n🎯 KEY FINDINGS:")
    print(f"   • Missing Data: {df.isnull().sum().sum():,} missing values ({(df.isnull().sum().sum() / (len(df) * len(df.columns))) * 100:.2f}%)")
    print(f"   • Duplicate Records: {df.duplicated().sum():,} duplicates ({df.duplicated().sum() / len(df) * 100:.2f}%)")
    
    # Calculate some key statistics
    numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    if numerical_cols:
        # Calculate strong correlations
        corr_matrix = df[numerical_cols].corr()
        strong_corr_count = 0
        for i in range(len(corr_matrix.columns)):
            for j in range(i+1, len(corr_matrix.columns)):
                if abs(corr_matrix.iloc[i, j]) > 0.5:
                    strong_corr_count += 1
        print(f"   • Strong Correlations: {strong_corr_count} pairs with |r| > 0.5")
        
        # Calculate outlier summary
        total_outliers = 0
        for col in numerical_cols:
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1
            outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
            total_outliers += len(outliers)
        
        print(f"   • Potential Outliers: {total_outliers:,} data points ({total_outliers / len(df) * 100:.2f}%)")
    
    print(f"\n💡 RECOMMENDATIONS FOR ANALYSIS:")
    print(f"   1. 🔍 Data Cleaning:")
    
    if df.isnull().sum().sum() > 0:
        print(f"      • Address missing values before analysis")
        print(f"      • Consider imputation strategies based on missingness patterns")
    else:
        print(f"      • ✅ No missing values - dataset is complete")
    
    if df.duplicated().sum() > 0:
        print(f"      • Review and remove duplicate records")
    else:
        print(f"      • ✅ No duplicates found")
    
    print(f"   2. 📊 Feature Engineering:")
    if len(categorical_cols) > 0:
        print(f"      • Consider encoding categorical variables for ML models")
        print(f"      • Analyze category frequencies for potential grouping")
    
    if len(numerical_cols) > 0:
        print(f"      • Check for normalization/standardization needs")
        print(f"      • Consider transformation for skewed distributions")
    
    print(f"   3. 🎯 Analysis Focus Areas:")
    print(f"      • Mental health prevalence analysis by demographics")
    print(f"      • Risk factor identification and correlation analysis")
    print(f"      • Temporal patterns and trends")
    print(f"      • Academic performance relationships")
    
    print(f"   4. 📈 Modeling Considerations:")
    if len(numerical_cols) > 0:
        print(f"      • Evaluate feature scaling requirements")
        print(f"      • Handle outliers appropriately for model type")
    
    print(f"      • Consider class imbalance in target variables")
    print(f"      • Plan train/validation/test splits appropriately")
    
    print(f"\n🏆 DATA QUALITY ASSESSMENT:")
    avg_completeness = ((len(df) * len(df.columns) - df.isnull().sum().sum()) / (len(df) * len(df.columns))) * 100
    
    if avg_completeness >= 95:
        quality_status = "🟢 EXCELLENT - Dataset ready for analysis"
    elif avg_completeness >= 90:
        quality_status = "🟡 GOOD - Minor cleaning recommended"  
    elif avg_completeness >= 75:
        quality_status = "🟠 FAIR - Significant cleaning needed"
    else:
        quality_status = "🔴 POOR - Extensive preprocessing required"
    
    print(f"   {quality_status}")
    print(f"   Overall Completeness: {avg_completeness:.1f}%")
    
    print(f"\n" + "=" * 80)
    print(f"📋 EDA COMPLETE - Dataset ready for targeted analysis!")
    print(f"=" * 80)
    
else:
    print("❌ Cannot generate final summary - data not available")

print(f"\n🎉 Exploratory Data Analysis Complete!")
print(f"You now have comprehensive insights into your student mental health dataset.")
print(f"Use these findings to guide your subsequent analysis and modeling decisions.")